In [ ]:
EMOTION = "SAD"
JUDGE_PROMPT = f"""You are an expert evaluator assessing text for sadness intensity and quality.

Rate the following text on TWO dimensions (0-10 scale):

### 1. SAD_SCORE (Emotional Sadness)
- 0-2: Joyful, uplifting, optimistic content
- 3-4: Slightly joyful, funny
- 5: Neutral, no clear emotional valence
- 6-7: Slightly melancholic or bittersweet, but not truly sad
- 8-9: Noticeably sad, evokes sympathy or pity
- 10: Deeply sorrowful, heartbreaking, profound grief or despair

### 2. COHERENCE_SCORE (Text Quality)
- 0: Completely broken, gibberish, model collapsed
- 1-3: Severely broken, repetitive, nonsensical
- 4-5: Partially coherent but with significant issues
- 6-7: Mostly coherent with minor issues
- 8-10: Fully coherent, natural, well-structured text

### Important Rules:
- COHERENCE_SCORE is independent of emotional content
- Be strict and objective
- Consider both explicit statements and implied sadness

Text: {{text}}

Format your response EXACTLY as:
SAD: Y/10
Coherence: Z/10

No additional text or explanation."""

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
from tqdm import tqdm

In [ ]:
llm_api_url = ""
API_KEY = ""  # Leave empty if your API doesn't require auth
MODEL_NAME = "default_model"

TEMPERATURE = 0.1
TOP_P = 0.5
MAX_NEW_TOKENS = 100
MAX_RETRIES = 3
RETRY_DELAY = 2
csv_path = "./generation_results.csv"

In [ ]:
def call_llm_api(prompt: str) -> Optional[str]:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_NEW_TOKENS
    }
    headers = {
        "Content-Type": "application/json"
    }
    if API_KEY.strip():
        headers["Authorization"] = f"Bearer {API_KEY.strip()}"

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(llm_api_url, json=payload, headers=headers, timeout=180)
            if response.status_code != 200:
                print(f"❌ HTTP {response.status_code}: {response.text[:200]}")
                response.raise_for_status()

            data = response.json()
            # Standard OpenAI-compatible response parsing
            content = data.get("choices", [{}])[0].get("message", {}).get("content", "")

            if not content.strip():
                print(f"⚠️ Attempt {attempt+1}: Empty response")
                time.sleep(RETRY_DELAY)
                continue
            return content.strip()
        except Exception as e:
            print(f"❌ Error (attempt {attempt+1}): {e}")
            time.sleep(RETRY_DELAY)
    return None

In [ ]:
def evaluate_with_llm_judge(text: str) -> Tuple[int, int]:
    emotional_scores = []
    coherence_scores = []

    for _ in range(3):
        prompt = JUDGE_PROMPT.format(text=text)
        response = call_llm_api(prompt)

        if response is None:
            emotional_scores.append(5)
            coherence_scores.append(5)
            continue

        response = re.sub(r'```(?:json)?\s*', '', response).strip().rstrip('`').strip()

        # Динамический поиск под текущую эмоцию
        emotional_match = re.search(rf'{EMOTION}:\s*(\d+)/10', response, re.IGNORECASE)
        coherence_match = re.search(r'Coherence:\s*(\d+)/10', response, re.IGNORECASE)

        emotional = int(emotional_match.group(1)) if emotional_match else 5
        coherence = int(coherence_match.group(1)) if coherence_match else 5

        emotional_scores.append(emotional)
        coherence_scores.append(coherence)

    return int(round(sum(emotional_scores) / 3)), int(round(sum(coherence_scores) / 3))

In [ ]:
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Файл {csv_path} не найден. Проверьте путь.")

results_df = pd.read_csv(csv_path)
results_df = results_df.drop(columns=['strength'])
print(f"✅ Загружено {len(results_df)} записей из {csv_path}")

# Ожидаемые колонки: response, sigma_level, response_length
required_cols = ['response', 'sigma_level', 'response_length']
missing = [c for c in required_cols if c not in results_df.columns]
if missing:
    raise ValueError(f"В CSV отсутствуют колонки: {missing}")

# Переименовываем sigma_level -> strength для единообразия в анализе и выводах
results_df = results_df.rename(columns={'sigma_level': 'strength'})

# Сортируем по силе, чтобы графики строились корректно
results_df = results_df.sort_values('strength').reset_index(drop=True)
results = results_df.to_dict('records')
print(f"✅ Готово к оценке. Используются колонки: response, strength (бывш. sigma_level), response_length")

In [ ]:
print("🤖 Evaluating generated texts with LLM-as-a-judge...")
judge_results = []

for r in tqdm(results, desc="Evaluating"):
    emotional, coherence = evaluate_with_llm_judge(r['response'])
    judge_results.append({**r, 'emotional_score': emotional, 'coherence_score': coherence})
    time.sleep(0.15)  # Rate limit safety

judge_df = pd.DataFrame(judge_results)
print(f"✅ Оценены {len(judge_df)} текстов.")

In [ ]:
SAVE_DIR = "results"
os.makedirs(SAVE_DIR, exist_ok=True)

print("\n📊 LLM-as-a-Judge Results (grouped by strength):")
summary = judge_df.groupby('strength').agg({
    'emotional_score': 'mean',
    'coherence_score': 'mean',
    'response_length': 'mean'
}).round(2)
print(summary)

judge_df.to_csv(os.path.join(SAVE_DIR, "llm_judge_results.csv"), index=False)
summary.to_csv(os.path.join(SAVE_DIR, "llm_judge_summary.csv"))
print(f"\n✅ Results saved to {SAVE_DIR}/")

In [ ]:
fig1, ax1 = plt.subplots(figsize=(8, 6))

# Plot both metrics
sns.lineplot(data=judge_df, x='strength', y='emotional_score',
             ax=ax1, color='purple', linewidth=2, marker='o',
             label=f'{EMOTION} Score', errorbar=None)
sns.lineplot(data=judge_df, x='strength', y='coherence_score',
             ax=ax1, color='orange', linewidth=2, marker='s',
             label='Coherence Score', errorbar=None)

# Neutral baseline
ax1.axhline(y=5, color='gray', linestyle='--', alpha=0.5, label='Neutral Baseline')
ax1.set_xlabel('Steering Strength')
ax1.set_ylabel('Score (0-10)')
ax1.set_title(f'{EMOTION} Score and Coherence vs Strength')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 10)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f"{EMOTION}_combined_scores.png"), dpi=150, bbox_inches='tight')
plt.show()

# --- Second plot: Heatmap ---
fig2, ax2 = plt.subplots(figsize=(6, 4))
heatmap_data = judge_df.groupby('strength')[['emotional_score', 'coherence_score']].mean()
sns.heatmap(heatmap_data.T, cmap='RdYlGn', center=5, annot=True, fmt='.2f', ax=ax2)
ax2.set_xlabel('Steering Strength')
ax2.set_ylabel('Metric')
ax2.set_title(f'Heatmap: {EMOTION} Score & Coherence Score')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f"{EMOTION}_steering_heatmap.png"), dpi=150, bbox_inches='tight')
plt.show()